In [ ]:
import tables
from ctapipe.visualization import CameraDisplay
from ctapipe.coordinates import EngineeringCameraFrame
from ctapipe_io_lst import LSTEventSource
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# This is just to obtain the information on the LST camera geometry:
# (you can ignore warnings on missing reference metadata)
sa = LSTEventSource.create_subarray(tel_id=1)
focal_length = sa.tel[1].optics.equivalent_focal_length
camera_geom = sa.tel[1].camera.geometry

In [ ]:
# Load one DL1 data check file:
a = tables.open_file("/data/cta/users-ifae/moralejo/CTA/summer_student_2026/datacheck/datacheck_dl1_LST-1.Run24704.h5")

In [ ]:
# Have a look at the file contents:
a.root.dl1datacheck   #AGL: it opens the main ('root') folder and then the subfolder 'dl1datacheck'
#
# The cosmics, flatfield and pedestals tables contain one entry per SUBRUN (which corresponds to ~8 to tens of seconds of observations, depending on epoch,
# zenith angle...). The quantities in the table are computed with all events (of the given type) in the subrun
#
# cosmics: table made using events triggered by the "physics trigger", i.e. because an increase of light was detected somewhere in the camera. Mostly Air Showers, but 
# it can include anything else: car flashes, satellites, meteors... This is the table most interesting to look for anomalous behaviour. 
#
# flatfield: made from interleaved flatfield events (in which mthe camera is illuminated uniformly with a fast flasher)
#
# pedestals: made from interleaved pedestal events (an event is recorded at a random instant), which contain only noise fluctuations



In [ ]:
# NOTES: 
#alt az cambian porque el tel se mueve en los subruns, ra dec no
# num clean events vs num events per subrun
# dragon time: UTC for 50 events through the run podemos calcular el tiempo medio de un subrun
# lo da en unix time , podemos convertirlo a tiempo normal con datetime y poner que lo convierta a  UTC timezome
#hists: deltat, npixels, nislands(islas de pixels que sobreviben el cleaning) pixelschargespectrum (cargas de los pixels en total), 
#tgrad ajuste dlineal 
#nearby stars: si cada pixel tiene una estrella cerca 

#cleaning: 8 p.e. 
#el tiempo de los pixels es escogido en los picos de voltaje, para pocos pe es poco riguroso porque fluctua mucho  

#rel time: flatfield events, para comparar entre pixels 

In [ ]:
# Each table has one entry per subrun:
a.root.dl1datacheck.cosmics.col('subrun_index')  

#AGL: .col('column') extracts a column within the selected file

In [ ]:
# Duration of the subruns (useful e.g. to convert event numbers to rates):
a.root.dl1datacheck.cosmics.col('elapsed_time')

In [ ]:
# Some of the table columns are named "hist_*". Those contain histogrammed quantities. For example, "intensity" is the sum of the charge (=light) in all pixels 
# of an event which contain a significant charge above the fluctuations (= pixels which survive a process called "image cleaning")
a.root.dl1datacheck.cosmics.col('hist_intensity').shape
# Shape is (number_of_subruns, number_of_bins). The array content is the number of events in each of the 100 bins

In [ ]:
# The bin edges (array of length nbins+1), intensity values in photoelectrons (p.e.) in this case, are stored here:
a.root.dl1datacheck.histogram_binning.col('hist_intensity')[0]   

#AGL: this gives the 'boundaries' of each bar of the intensities histogram

In [ ]:
a.root.dl1datacheck.cosmics.col('hist_intensity').shape  # (n_subruns, n_bins)

In [ ]:
# Some of the quantities contain values "per pixel". There are 1855 pixels in the LST1 camera. An example is how many of the centers of gravity of the images 
# (i.e. the charge-weighted mean position of the pixels surviving cleaning) fall within each pixel.  
a.root.dl1datacheck.cosmics.col('cog_within_pixel').shape  # (n_subruns, n_pixels)

#AGL: the CoG is like the "center" of mass but with the charge induced by the light arriving at each pixel
#     the column 'cog_within_pixel' counts the number of times the CoG of the event has fallen into that pixel along the subrun

In [ ]:
# Let's plot the mean value of that for the first 8 subruns:
camdisplay = CameraDisplay(camera_geom.transform_to(EngineeringCameraFrame()),
                           norm='lin', title='# of events with CoG in pixel', 
                           image=np.mean(a.root.dl1datacheck.cosmics.col('cog_within_pixel')[:8,:], axis=0))

#AGL: this takes the number of times the CoG has fallen in each pixel for the first 8 subruns, does the mean, and plots it
camdisplay.autoupdate = True
camdisplay.add_colorbar()
plt.show()

#Poisson

In [ ]:
# Now let's plot it for the ninth subrun (index=8)...

camdisplay = CameraDisplay(camera_geom.transform_to(EngineeringCameraFrame()),
                          norm='lin', title='# of events with CoG in pixel', 
                           image=a.root.dl1datacheck.cosmics.col('cog_within_pixel')[8,:])

#AGL: during the elapse time of the 9-th subrun, the number of cog within pixel is counted, this is what this plot shows

#AGL: we can see, for instance, that in the brightest pixels the CoG has fallen approx 1200 time within the subrun

camdisplay.autoupdate = True
camdisplay.add_colorbar()
plt.show()

### Overview of all the parameters available:
a.root.dl1datacheck.cosmics

In [ ]:
# Now let's plot the intensity histograms for all subruns
# First the binning:
intensity_bins = a.root.dl1datacheck.histogram_binning.col('hist_intensity')[0]

#AGL: even though 'hist_' data are histogrammed quantities, to deal with the binning we need to look inside the 'histogram_binning' folder

In [ ]:
# bin centers:
intensity = 0.5*(intensity_bins[1:]+intensity_bins[:-1])  

#AGL: we take the superior [:1] limit and the inferior [:-1] limit and find the middle point

In [ ]:
plt.figure(figsize=(14,5))
for subrun in range(a.root.dl1datacheck.cosmics.col('hist_intensity').shape[0]):  #AGL: .shape[0] gives the number of subruns
    time = a.root.dl1datacheck.cosmics.col('elapsed_time')[subrun]
    #plt.plot(3**subrun*intensity, a.root.dl1datacheck.cosmics.col('hist_intensity')[subrun]/time)
    plt.plot(intensity, a.root.dl1datacheck.cosmics.col('hist_intensity')[subrun]/time)

#AGL: with this, we intend to draw the intensity histogram for each subrun but using 

#     x = center of the bin
#     y = value of the histogram column

# instead of using columns and bins. Hence we get a curve with the light intensity (rate, since it's /time)
# for each subrun (color) 

# For this dcheck we can see that the Intensity profile stands out for a given subrun, indicating the presence 
# of something brightest crossing the Tel FoV

plt.xlabel('Intensity')
plt.ylabel('Events/s')
plt.grid()
plt.xscale('log')
plt.show()

#Poisson: largo intervalo: asignar los eventos random en el intervalo de tiempo. hacer un histograma con estos valores en 1 millon segundo, mandamos 3 millones sucesos mu=3
#sucesos independientes pero con rate medio constante 
# si graficamos un deltat, tiene que salir una distribucion exponencial
# mu grande -> gaussiana aprox 
#la pendiente es mu, rate/s, para el satelite prob tecnico del tel

In [ ]:
# deltat is the time difference in milliseconds between a given event and the previous one.
# It is obviously related to the event rate. 
# In a Poisson process (like the detection of air showers) it should follow an exponential behaviour...


#AGL: if the event rate is hight, deltat will be smaller. 
deltat_bins = a.root.dl1datacheck.histogram_binning.col('hist_delta_t')[0]
deltat = 0.5*(deltat_bins[1:]+deltat_bins[:-1])

#AGL: That deltat histogram does not follow an exponential also indicates the presence of smth different to air showers

In [ ]:
plt.figure(figsize=(10,4))
for subrun in range(a.root.dl1datacheck.cosmics.col('hist_delta_t').shape[0]):
    time = a.root.dl1datacheck.cosmics.col('elapsed_time')[subrun]
    plt.plot(deltat, a.root.dl1datacheck.cosmics.col('hist_delta_t')[subrun]/time)
plt.grid()
plt.xlabel('delta_t (ms)')
plt.yscale('log')
plt.show()